In [51]:
"""S3 utils"""
import boto3
from typing import Any
import csv
import smart_open
from datetime import datetime
import polars as pl

S3 = boto3.Session(profile_name="platform-developer").client("s3")
TRANSPORT_PARAMS = {"client": S3}

def df_from_s3_parquet(s3_uri: str) -> pl.DataFrame:
    with smart_open.open(s3_uri, "rb", transport_params=TRANSPORT_PARAMS) as f:
        return pl.read_parquet(f)


def get_csv_from_s3(s3_uri: str) -> list[Any]:
    with smart_open.open(s3_uri, "r", transport_params=TRANSPORT_PARAMS) as f:
        return list(csv.DictReader(f))


def list_s3_uris(bucket_name: str, prefix: str, suffix: str) -> list[str]:
    paginator = S3.get_paginator("list_objects_v2")
    page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    files = []
    for page in page_iterator:
        for item in page.get("Contents", []):
            if item["Key"].endswith(suffix):
                files.append(f"s3://{bucket_name}/{item['Key']}")

    return files

def list_incremental_windows(bucket_name: str, prefix: str, range_start: datetime | None = None, range_end: datetime | None = None) -> list[tuple[datetime, datetime]]:
    windows = []
    paginator = S3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix, Delimiter="/"):
        for common_prefix in page["CommonPrefixes"]:
            full_prefix = common_prefix["Prefix"]
            window = full_prefix[len(prefix):].rstrip("/")
            
            try:
                window_start, window_end = strp_window(window)
            except ValueError:
                print(f"Could not parse prefix '{window}' into a time window.")
                continue

            starts_after_range_start = not range_start or window_start >= range_start
            ends_before_range_end = not range_end or window_end <= range_end
            if starts_after_range_start and ends_before_range_end:
                windows.append((window_start, window_end))
        
    return windows


In [38]:
"""Incremental window utils"""

def strf_window(window: tuple[datetime, datetime]) -> str:
    return f"{window[0].strftime('%Y%m%dT%H%M')}-{window[1].strftime('%Y%m%dT%H%M')}"

def strp_window(window: str) -> tuple[datetime, datetime]:
    raw_start, raw_end = window.split("-")
    window_start = datetime.strptime(raw_start, "%Y%m%dT%H%M")
    window_end = datetime.strptime(raw_end, "%Y%m%dT%H%M")
    return window_start, window_end

In [52]:
from utils.types import TransformerType, EntityType, CatalogueTransformerType
from pydantic import BaseModel
from concurrent.futures import ThreadPoolExecutor
from itertools import chain
from typing import Literal

S3_PARALLELISM = 10
GRAPH_BUCKET_NAME = "wellcomecollection-catalogue-graph"
PIPELINE_DATE = "2025-10-02"


class BaseS3Debugger(BaseModel):
    bucket_name: str = GRAPH_BUCKET_NAME
    pipeline_date: str = PIPELINE_DATE

    @property
    def service_prefix(self) -> str:
        raise NotImplementedError()

    def get_uris_for_window(window: tuple[datetime, datetime]) -> list[str]:
        raise NotImplementedError()
    
    def get_incremental_file(s3_uri: str) -> pl.DataFrame:
        raise NotImplementedError()

    @property
    def windows_prefix(self) -> str:
        return f"{self.service_prefix}/{self.pipeline_date}/windows"

    def get_single_window_df(self, window: tuple[datetime, datetime]):
        dfs = []
        for s3_uri in self.get_uris_for_window(window):
            try:
                df = self.get_incremental_file(s3_uri)
                if len(df) > 0:
                    dfs.append(df)
            except OSError as e:
                print(e)
                print(f"File '{s3_uri}' does not exist")

        if len(dfs) == 0:
            return pl.DataFrame()

        df = pl.concat(dfs)
        df = df.with_columns(pl.lit(window[0]).alias("window_start"))
        df = df.with_columns(pl.lit(window[1]).alias("window_end"))
        return df

    def dataframe_from_incremental_files(self, range_start: datetime | None = None, range_end: datetime | None = None):
        windows = list_incremental_windows(self.bucket_name, f"{self.windows_prefix}/", range_start, range_end)
        print(f"Will process {len(windows):,} {self.service_prefix} time windows. Pipeline date: {self.pipeline_date}.")
        
        with ThreadPoolExecutor(max_workers=S3_PARALLELISM) as executor:
            dfs = executor.map(self.get_single_window_df, windows)

        non_empty_dfs = [df for df in dfs if len(df) > 0]
        if not non_empty_dfs:
            return pl.DataFrame()

        full_df = pl.concat(non_empty_dfs)
        print(f"Retrieved {len(full_df):,} rows")
    
        return full_df.sort("window_end")        

class BulkLoaderDebugger(BaseS3Debugger):
    transformer_type: TransformerType
    entity_type: EntityType

    @property
    def service_prefix(self) -> str:
        return "graph_bulk_loader"

    @property
    def file_name(self) -> str:
        return f"{self.transformer_type}__{self.entity_type}.csv"

    @property
    def full_reindex_uri(self) -> str:
        return f"s3://{self.bucket_name}/{self.service_prefix}/{self.pipeline_date}/{self.file_name}"

    def get_uris_for_window(self, window: tuple[datetime, datetime]) -> list[str]:
        return [f"s3://{self.bucket_name}/{self.windows_prefix}/{strf_window(window)}/{self.file_name}"]

    def get_incremental_file(self, s3_uri: str) -> pl.DataFrame:
        return pl.DataFrame(get_csv_from_s3(s3_uri))

    def dataframe_from_full_reindex_file(self):
        df = pl.DataFrame(get_csv_from_s3(self.full_reindex_uri))
        print(f"Retrieved a full reindex bulk load file storing {len(df):,} rows")
        return df


class GraphRemoverIncrementalDebugger(BaseS3Debugger):
    transformer_type: CatalogueTransformerType
    entity_type: EntityType

    @property
    def service_prefix(self) -> str:
        return "graph_remover_incremental"

    @property
    def file_name(self) -> str:
        return f"{self.transformer_type}__{self.entity_type}.parquet"

    def get_uris_for_window(self, window: tuple[datetime, datetime]) -> list[str]:
        return [f"s3://{self.bucket_name}/{self.windows_prefix}/{strf_window(window)}/deleted_ids/{self.file_name}"]

    def get_incremental_file(self, s3_uri: str):
        df = df_from_s3_parquet(s3_uri)
        if len(df) > 0:
            df.columns = ["id"]
        return df


class IngestorDebugger(BaseS3Debugger):
    ingestor_type: Literal["works", "concepts"]
    index_date: str

    @property
    def service_prefix(self) -> str:
        return f"ingestor_{self.ingestor_type}"

    @property
    def windows_prefix(self) -> str:
        return f"{self.service_prefix}/{self.pipeline_date}/{self.index_date}"
    
    def get_uris_for_window(self, window: tuple[datetime, datetime]) -> str:
        prefix = f"{self.windows_prefix}/{strf_window(window)}/"
        return list_s3_uris(self.bucket_name, prefix, ".parquet")

    def get_incremental_file(self, s3_uri: str):
        return df_from_s3_parquet(s3_uri)

In [18]:
"""Example bulk loader debugger"""
debugger = BulkLoaderDebugger(
    transformer_type="catalogue_concepts",
    entity_type="nodes"
)
df = debugger.dataframe_from_incremental_files()
df.filter(pl.col(":ID") == "kwyaf9w7")

Will process 15,186 graph_bulk_loader time windows. Pipeline date: 2025-10-02.
File 's3://wellcomecollection-catalogue-graph/graph_bulk_loader/2025-10-02/windows/20260218T1550-20260218T1600/catalogue_concepts__nodes.csv' does not exist
Retrieved 4,582,129 rows


:ID,:LABEL,id:String,label:String,source:String,window_start,window_end
str,str,str,str,str,datetime[μs],datetime[μs]
"""kwyaf9w7""","""Concept""","""kwyaf9w7""","""Field, Nicola""","""label-derived""",2025-11-25 15:45:00,2025-11-25 16:00:00
"""kwyaf9w7""","""Concept""","""kwyaf9w7""","""Field, Nicola""","""label-derived""",2025-11-25 16:00:00,2025-11-25 16:15:00
"""kwyaf9w7""","""Concept""","""kwyaf9w7""","""Field, Nicola""","""label-derived""",2026-02-04 10:00:00,2026-02-04 10:15:00
"""kwyaf9w7""","""Concept""","""kwyaf9w7""","""Field, Nicola""","""label-derived""",2026-02-05 13:30:00,2026-02-05 13:45:00
"""kwyaf9w7""","""Concept""","""kwyaf9w7""","""Field, Nicola""","""label-derived""",2026-02-13 14:45:00,2026-02-13 15:00:00
"""kwyaf9w7""","""Concept""","""kwyaf9w7""","""Field, Nicola""","""label-derived""",2026-03-17 14:45:00,2026-03-17 15:00:00
"""kwyaf9w7""","""Concept""","""kwyaf9w7""","""Field, Nicola""","""label-derived""",2026-03-18 14:00:00,2026-03-18 14:15:00
"""kwyaf9w7""","""Concept""","""kwyaf9w7""","""Field, Nicola""","""label-derived""",2026-03-18 16:00:00,2026-03-18 16:15:00


In [15]:
"""Example graph remover incremental debugger"""
debugger = GraphRemoverIncrementalDebugger(
    transformer_type="catalogue_concepts",
    entity_type="nodes"
)
range_start = datetime(2025, 10, 5)
range_end = datetime(2025, 10, 20)
df = debugger.dataframe_from_incremental_files(range_start, range_end)
df.filter(pl.col("id") == "snb65nn6")

Will process 325 graph_remover_incremental time windows. Pipeline date: 2025-10-02.
Retrieved 14 rows


In [66]:
"""Example ingestor debugger"""
debugger = IngestorDebugger(
    ingestor_type="concepts",
    index_date="2026-03-03"
)

df = debugger.dataframe_from_incremental_files()
df.filter(pl.col("query").struct.field("id") == "taythpbh")

Could not parse prefix '20260303T1527' into a time window.
Could not parse prefix 'dev' into a time window.
Will process 1,366 ingestor_concepts time windows. Pipeline date: 2025-10-02.
Retrieved 1,380,547 rows


query,display,window_start,window_end
struct[5],struct[10],datetime[μs],datetime[μs]
"{""taythpbh"",[{""D012852"",""nlm-mesh""}],""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],""Subject""}","{""taythpbh"",[{""D012852"",""Identifier"",{""nlm-mesh"",""IdentifierType"",""Medical Subject Headings (MeSH) identifier""}}],""Sinusitis"",""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],{""Inflammation of the mucous membrane that lines the sinuses resulting in symptoms"",""wikidata"",""https://www.wikidata.org/wiki/Q183344""},""Subject"",{[],[],[{""Inflammation"",""czhbuy4h"",null,""Subject""}, {""Paranasal sinuses - Diseases"",""bvdnmueg"",null,""Subject""}],[{""Frontal Sinusitis"",""b79y3zre"",null,""Subject""}, {""Maxillary Sinusitis"",""d9dnj3aa"",null,""Subject""}],[],[],[{""Pharmaceutical Preparations"",""apcmv97q"",null,""Subject""}, {""Drug Industry"",""ak8zkxyz"",null,""Subject""}, … {""Cuba"",""dena7nx4"",null,""Place""}],[]},[""kx3xg2cs"", ""mk87qxbk"", … ""zp5hw4ab""],[]}",2026-03-11 17:15:00,2026-03-11 17:30:00
"{""taythpbh"",[{""D012852"",""nlm-mesh""}],""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],""Subject""}","{""taythpbh"",[{""D012852"",""Identifier"",{""nlm-mesh"",""IdentifierType"",""Medical Subject Headings (MeSH) identifier""}}],""Sinusitis"",""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],{""Inflammation of the mucous membrane that lines the sinuses resulting in symptoms"",""wikidata"",""https://www.wikidata.org/wiki/Q183344""},""Subject"",{[],[],[{""Inflammation"",""czhbuy4h"",null,""Subject""}, {""Paranasal sinuses - Diseases"",""bvdnmueg"",null,""Subject""}],[{""Frontal Sinusitis"",""b79y3zre"",null,""Subject""}, {""Maxillary Sinusitis"",""d9dnj3aa"",null,""Subject""}],[],[],[{""Pharmaceutical Preparations"",""apcmv97q"",null,""Subject""}, {""Drug Industry"",""ak8zkxyz"",null,""Subject""}, … {""Paranasal Sinuses"",""bnn342un"",null,""Subject""}],[]},[""kx3xg2cs"", ""mk87qxbk"", … ""zp5hw4ab""],[]}",2026-03-11 18:15:00,2026-03-11 18:30:00
"{""taythpbh"",[{""D012852"",""nlm-mesh""}],""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],""Subject""}","{""taythpbh"",[{""D012852"",""Identifier"",{""nlm-mesh"",""IdentifierType"",""Medical Subject Headings (MeSH) identifier""}}],""Sinusitis"",""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],{""Inflammation of the mucous membrane that lines the sinuses resulting in symptoms"",""wikidata"",""https://www.wikidata.org/wiki/Q183344""},""Subject"",{[],[],[{""Inflammation"",""czhbuy4h"",null,""Subject""}, {""Paranasal sinuses - Diseases"",""bvdnmueg"",null,""Subject""}],[{""Frontal Sinusitis"",""b79y3zre"",null,""Subject""}, {""Maxillary Sinusitis"",""d9dnj3aa"",null,""Subject""}],[],[],[{""Pharmaceutical Preparations"",""apcmv97q"",null,""Subject""}, {""Drug Industry"",""ak8zkxyz"",null,""Subject""}, … {""Cuba"",""dena7nx4"",null,""Place""}],[]},[""kx3xg2cs"", ""mk87qxbk"", … ""zp5hw4ab""],[]}",2026-03-11 19:15:00,2026-03-11 19:30:00
"{""taythpbh"",[{""D012852"",""nlm-mesh""}],""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],""Subject""}","{""taythpbh"",[{""D012852"",""Identifier"",{""nlm-mesh"",""IdentifierType"",""Medical Subject Headings (MeSH) identifier""}}],""Sinusitis"",""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],{""Inflammation of the mucous membrane that lines the sinuses resulting in symptoms"",""wikidata"",""https://www.wikidata.org/wiki/Q183344""},""Subject"",{[],[],[{""Inflammation"",""czhbuy4h"",null,""Subject""}, {""Paranasal sinuses - Diseases"",""bvdnmueg"",null,""Subject""}],[{""Frontal Sinusitis"",""b79y3zre"",null,""Subject""}, {""Maxillary Sinusitis"",""d9dnj3aa"",null,""Subject""}],[],[],[{""Pharmaceutical Preparations"",""apcmv97q"",null,""Subject""}, {""Drug Industry"",""ak8zkxyz"",null,""Subject""}, … {""Cuba"",""dena7nx4""

In [65]:
df

query,display,window_start,window_end
struct[5],struct[10],datetime[μs],datetime[μs]
"{""taythpbh"",[{""D012852"",""nlm-mesh""}],""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],""Subject""}","{""taythpbh"",[{""D012852"",""Identifier"",{""nlm-mesh"",""IdentifierType"",""Medical Subject Headings (MeSH) identifier""}}],""Sinusitis"",""Sinusitis"",[""Infection, Sinus"", ""Infections, Sinus"", … ""Sinusitis""],{""Inflammation of the mucous membrane that lines the sinuses resulting in symptoms"",""wikidata"",""https://www.wikidata.org/wiki/Q183344""},""Subject"",{[],[],[{""Inflammation"",""czhbuy4h"",null,""Subject""}, {""Paranasal sinuses - Diseases"",""bvdnmueg"",null,""Subject""}],[{""Frontal Sinusitis"",""b79y3zre"",null,""Subject""}, {""Maxillary Sinusitis"",""d9dnj3aa"",null,""Subject""}],[],[],[{""Pharmaceutical Preparations"",""apcmv97q"",null,""Subject""}, {""Drug Industry"",""ak8zkxyz"",null,""Subject""}, … {""Paranasal Sinuses"",""bnn342un"",null,""Subject""}],[]},[""kx3xg2cs"", ""mk87qxbk"", … ""zp5hw4ab""],[]}",2026-03-15 14:45:00,2026-03-15 15:00:00
